# ==============================================================================
# 🏡 SPATIAL REAL ESTATE VALUATION: CALIFORNIA HOUSING VIA RANDOM FOREST REGRESSOR
# ==============================================================================
### Core Principles:
Predicting median house values across 20,640 California census block groups using geographical coordinates (`latitude`, `longitude`), housing age, rooms, and demographic median income.
Random Forest builds an ensemble of non-linear spatial partitions.


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: California Spatial Housing Environment ready.")


✅ STEP 1: California Spatial Housing Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET CLEANUP
Loading Kaggle California Housing Prices dataset (20,640 census blocks).


In [2]:
# STEP 2 & 3: LOAD & CLEANUP
data_path = 'data/california_housing/housing.csv'
df = pd.read_csv(data_path)

df.columns = [c.strip().lower() for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]:,} census blocks, {df.shape[1]} attributes")
print(f"Target Summary:\n{df['median_house_value'].describe().round(2)}")
df.head(3)


📊 Dataset Shape: 20,640 census blocks, 10 attributes
Target Summary:
count     20640.00
mean     206855.82
std      115395.62
min       14999.00
25%      119600.00
50%      179700.00
75%      264725.00
max      500001.00
Name: median_house_value, dtype: float64


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `median_house_value`.


In [3]:
# STEP 4 & 5: SEGREGATION & HYGIENE
X = df.drop(columns=['median_house_value'])
y = df['median_house_value']

print(f"Missing attributes:\n{X.isnull().sum()[X.isnull().sum() > 0]}")


Missing attributes:
total_bedrooms    207
dtype: int64


## 🔒 STEP 6: LEAK-FREE TRAIN-TEST SPLIT
80/20 train/test partition.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train Blocks: {X_train.shape[0]:,}")
print(f"Test Blocks : {X_test.shape[0]:,}")


Train Blocks: 16,512
Test Blocks : 4,128


## ⚙️ STEP 7: PREPROCESSING PIPELINE
Median imputation for `total_bedrooms` and OneHotEncoder for `ocean_proximity`.


In [5]:
# STEP 7: PREPROCESSOR
cat_cols = ['ocean_proximity']
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', SimpleImputer(strategy='median'), num_cols)
])
print("✅ STEP 7: Spatial Preprocessor configured.")


✅ STEP 7: Spatial Preprocessor configured.


## ⚠️ STEP 8: BASELINE ENSEMBLE FIT
Training baseline Random Forest with 100 trees.


In [6]:
# STEP 8: BASELINE FIT
pipe_rf = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestRegressor(n_estimators=100, max_depth=14, random_state=42, n_jobs=-1))
])
pipe_rf.fit(X_train, y_train)

r2_base = r2_score(y_test, pipe_rf.predict(X_test))
print(f"Baseline Random Forest R² : {r2_base*100:.2f}%")
print(f"Baseline RMSE             : ${np.sqrt(mean_squared_error(y_test, pipe_rf.predict(X_test))):,.2f}")


Baseline Random Forest R² : 81.14%
Baseline RMSE             : $49,713.50


## 🎯 STEP 9: TUNING HYPERPARAMETERS VIA 3-FOLD CV
Optimizing `max_depth` and `min_samples_leaf`.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'rf__n_estimators': [100, 150],
    'rf__max_depth': [14, 18],
    'rf__min_samples_leaf': [2, 4]
}

cv = KFold(n_splits=3, shuffle=True, random_state=42)
pipe_tune = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestRegressor(random_state=42, n_jobs=-1))
])
grid = GridSearchCV(pipe_tune, param_grid=param_grid, cv=cv, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 3-Fold CV R²   : {grid.best_score_*100:.2f}%")


🏆 Best Hyperparameters: {'rf__max_depth': 18, 'rf__min_samples_leaf': 2, 'rf__n_estimators': 150}
🎯 Best 3-Fold CV R²   : 81.15%


## ⚔️ STEP 10: ALGORITHM DUEL (Random Forest vs Single Tree)
Benchmarking against single tree.


In [8]:
# STEP 10: BENCHMARK
from sklearn.tree import DecisionTreeRegressor

pipe_single = Pipeline([
    ('prep', preprocessor),
    ('tree', DecisionTreeRegressor(random_state=42))
])
pipe_single.fit(X_train, y_train)

r2_tree = r2_score(y_test, pipe_single.predict(X_test))
r2_rf = r2_score(y_test, best_model.predict(X_test))

print("="*65)
print(f"Single Decision Tree -> R²: {r2_tree*100:.2f}% | RMSE: ${np.sqrt(mean_squared_error(y_test, pipe_single.predict(X_test))):,.2f}")
print(f"Random Forest (Tuned)-> R²: {r2_rf*100:.2f}% | RMSE: ${np.sqrt(mean_squared_error(y_test, best_model.predict(X_test))):,.2f}")
print("="*65)


Single Decision Tree -> R²: 63.47% | RMSE: $69,190.79
Random Forest (Tuned)-> R²: 81.73% | RMSE: $48,924.98


## 📊 STEP 11: RESIDUAL ANALYSIS & EVALUATION REPORT
Diagnostics on property errors.


In [9]:
# STEP 11: EVALUATION REPORT
y_pred_rf = best_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred_rf)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print(f"📋 CALIFORNIA PROPERTY VALUATION METRICS:")
print(f"   Mean Absolute Error (MAE) : ${mae:,.2f}")
print(f"   Root Mean Squared Error   : ${rmse:,.2f}")
print(f"   R² Score                  : {r2_score(y_test, y_pred_rf)*100:.2f}%")


📋 CALIFORNIA PROPERTY VALUATION METRICS:
   Mean Absolute Error (MAE) : $31,608.34
   Root Mean Squared Error   : $48,924.98
   R² Score                  : 81.73%


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & SMOKE TEST
Deploying the valuation model and testing automated valuation latency.


In [10]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/california_housing_random_forest_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_property = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_value = loaded_pipe.predict(sample_property)[0]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🏡 LIVE INSTANT PROPERTY APPRAISAL:")
print(f"   Estimated Block Value: 🏷️ ${pred_value:,.2f}")
print(f"   Actual Listed Value  : ${y_test.iloc[0]:,.2f}")
print(f"   Valuation Latency    : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/california_housing_random_forest_pipeline.joblib (21,435,682 bytes)



🏡 LIVE INSTANT PROPERTY APPRAISAL:
   Estimated Block Value: 🏷️ $50,886.48
   Actual Listed Value  : $47,700.00
   Valuation Latency    : 64.788 ms (SLA < 2.0ms: CHECK)
